# Import Libaries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import KNNImputer

### Compass
Definition
- Protocol Number 6 = TCP
- Protocol Number 17 = UDP

# Read in Dataset

In [ ]:
original_df = pd.read_csv('../ddos_balanced/final_dataset.csv')
original_df.head()

--------
# Date preparation


In [ ]:
print("This Dataset has {} rows and {} columns".format(original_df.shape[0], original_df.shape[1]))

In [ ]:
original_df.info()

Note there are 2 object features here and to create the machine learning models later they have to be converted to ints or floats. 

In [ ]:
# Drop statstical columns such as columns containing 'Mean', 'Std', 'Min', or 'Max'
columns_to_drop = [col for col in original_df.columns if any(x in col for x in ["Mean", "Std", "Min", "Max","Avg"])]
df_clean_stats = original_df.drop(columns=columns_to_drop)

print("Remaining Columns:", df_clean_stats.columns)

In [ ]:
print("This Dataset has {} rows and {} columns".format(df_clean_stats.shape[0], df_clean_stats.shape[1]))

### Remove timestamp object column

In [ ]:
train_clean_stats_df = df_clean_stats.drop("Timestamp", axis='columns')

### Convert target from object to norminal data


In [ ]:
print(train_clean_stats_df["Label"].dtype)
print(train_clean_stats_df["Label"].unique())

In [ ]:
train_clean_stats_df["Label"] = train_clean_stats_df["Label"].map({"Benign": 0, "ddos": 1})

---
### Boundary checks
Finding numbers that are too large for float64

Select only numeric columns

In [ ]:
numeric_df = train_clean_stats_df.select_dtypes(include=[np.number])

Check for infinite values in numeric columns

In [ ]:
has_inf = np.isinf(numeric_df).any().any()
print("Contains infinite values:", has_inf)

Replace the infinite values in numeric columns with nan

In [ ]:
train_clean_stats_df[numeric_df.columns] = numeric_df.replace([np.inf, -np.inf], np.nan)

## Indentifying missing values

In [ ]:
print(train_clean_stats_df.isna()) 

In [ ]:
missing_values = train_clean_stats_df.isnull().sum()
# Filter out columns with null values
lis = missing_values[missing_values > 0]  
print(lis)

In [ ]:
# train_clean_stats_df.drop(columns=['Flow Byts/s', 'Flow Pkts/s'], inplace=True)

---
## Data Imputation
Using KNN to fill missing data for flow byts.

In [ ]:
knn_imputer = KNNImputer(n_neighbors=3000)

In [ ]:
cols_with_missing = lis.index

Apply KNN

In [ ]:
train_clean_stats_df[cols_with_missing] = knn_imputer.fit_transform(train_clean_stats_df[cols_with_missing])
train_df_clean = train_clean_stats_df

---
### Checking for data bias

In [ ]:
# Calculate class distribution
malign = train_df_clean[train_df_clean['Label'] == 1]
benign = train_df_clean[train_df_clean['Label'] == 0]

print(f'Malicious traffic: {len(malign)/len(train_df_clean):.2%}')
print(f'Benign traffic: {len(benign)/len(train_df_clean):.2%}')

### Visualising bias

In [ ]:
# Create visualization
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='Label', data=train_df_clean, palette='viridis')

# Customize plot
plt.title("Traffic Class Distribution", fontsize=14, pad=20)
plt.xlabel("Class Type", fontsize=12)
plt.ylabel("Percentage of Total", fontsize=12)

# Convert counts to percentages
total = len(train_df_clean)
for p in ax.patches:
    percentage = f'{100 * p.get_height()/total:.1f}%'
    x = p.get_x() + p.get_width()/2
    y = p.get_height() + total*0.01
    ax.annotate(percentage, (x, y), ha='center')

# Set proper labels
plt.xticks([0, 1], ['Benign', 'Malicious'], rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Note the dataset is extremly bias. The ML can be used to indetify, when an attack is happening. and if luckly it's a ftp or ssh brute force. Which currently no Ml sepcify in this detection. 

Most algo i read abt. is a general detection model but to better work with Fail2ban's jailing system, anti brute force detection a new ML detector is needed hence this project. 

Pairplot of: 
    Dst Port (Destination port)
    Protocol
    Flow Duration
    Tot Fwd Pkts (Total forward packets)
    Tot Bwd Pkts (Total backward packets)
    Label (Target)
which are the important feature based on Kaggle, the source of the Data. 

In [ ]:
print(train_df_clean['Protocol'].unique())

- 6 = TCP 
- 17 = UDP
- At this point no further information is gather about IP 0 therefore rows with 0 would be taken as N.A. and be dropped. 
- More reading have been done and Protocol 0 refers to HOPOPT which is a technique linked to IPV6. 
- Given that IPV6 would likely have better solutions to DDOS attacks this ML would not cover this area. 

In [ ]:
train_df_clean_dropIP = train_df_clean[train_df_clean['Protocol'] != 0]
print(train_df_clean_dropIP['Protocol'].unique())

In [ ]:
sns.pairplot(train_df_clean_dropIP,hue="Label",vars=['Protocol','Dst Port','Flow Duration'])

In [ ]:
train_df_clean_dropIP.columns

there are currently too many cols. therefore an random forest selection is used to pick out the top 30 

In [ ]:
train_df_clean_dropIP.info()

Features such as mean, averages, max and min are causing problems with the random forest therefore it will be deleted. 

In [ ]:
# # Drop columns containing 'Mean', 'Std', 'Min', or 'Max'
# columns_to_drop = [col for col in train_df_clean_dropIP.columns if any(x in col for x in ["Mean", "Std", "Min", "Max","Avg"])]
# train_df_clean_stats = train_df_clean_dropIP.drop(columns=columns_to_drop)

# print("Remaining Columns:", train_df_clean_stats.columns)

In [ ]:
train_df_clean_stats.info()

In [ ]:
print(train_df_clean_stats.iloc[:, [7, 16, 17, 18]])

---
## Feature Selection

PCA

In [ ]:
# Separate features and target variable
X = train_df_clean_stats.drop(columns=['Label'])  # Features
y = train_df_clean_stats['Label']  # Target variable

# Initialize Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# Select the top 30 important features
selector = SelectFromModel(rf, max_features=30, prefit=True)
selected_features = X.columns[selector.get_support()]

# Reduce the dataset to only the top 30 features
train_df_list = train_df_clean_stats[selected_features.tolist() + ['Label']]

# Print selected features
print("Top 30 selected features:", selected_features.tolist())

After the random forest only 11 feature met the requirements threshold and therfore these would be the selected features to train the model. 

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Separate features and target using the dataset with selected features
# train_df_list is assumed to have the selected features along with the 'Label' column
X = train_df_list.drop(columns=['Label'])
y = train_df_list['Label']

# Perform PCA to reduce the data to 2 components for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

# Plot the PCA results
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.7)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA of Top Selected Features")
plt.legend(*scatter.legend_elements(), title="Label")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the correlation matrix of the dataframe containing selected features and the target label
corr_matrix = train_df_list.corr()

# Set up the matplotlib figure
plt.figure(figsize=(12, 10))

# Create a heat map with annotations
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)

# Add title and labels
plt.title("Correlation Heatmap of Selected Features and Label")
plt.show()


---
## Exporting the selected features as the new training dataset

In [ ]:
train_df_list.to_csv("selected_features.csv", index=False)
print("CSV file saved as 'selected_features.csv'")